# Memory Management with Prompt Caching and Token-Aware Summarization

This notebook implements an intelligent memory management system that:
- Uses in-memory store for chat history
- Leverages OpenAI Prompt Caching for static system instructions
- Uses tiktoken to compute token limits
- Automatically summarizes conversations when token threshold is crossed
- Summarizes chat history every 5 messages

### Load Environment Variables

Load API keys from a `.env` file. Required for accessing OpenAI's API to generate responses and summaries in the memory management system.

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

### Install Required Packages

Install dependencies needed for memory management: `tiktoken` for accurate token counting, `langchain-openai` for LLM integration, and `langchain-core` for message handling. Run this cell once to set up dependencies.

In [ ]:
# Install required packages (run once if needed)
# !pip install tiktoken langchain-openai langchain-core

### Import Libraries

Import all necessary libraries for memory management: `tiktoken` for token counting, `deque` for efficient chat history storage, LangChain components for LLM interaction and message handling, and JSON for data serialization.

In [5]:
import tiktoken
from typing import List, Dict, Optional
from collections import deque
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import json

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


### Define MemoryManager Class

Create a comprehensive memory management system that:
- **Stores chat history** in memory using `deque` for efficient operations
- **Counts tokens** using `tiktoken` to track conversation length
- **Automatically summarizes** conversations when token threshold is exceeded or every N messages
- **Supports prompt caching** by maintaining consistent system prompts
- **Manages memory** by replacing old messages with summaries to stay within token limits

The class handles the complete lifecycle of conversation memory management.

In [9]:
class MemoryManager:
    """
    Intelligent memory management system with:
    - In-memory chat history storage
    - Token counting using tiktoken
    - Automatic summarization when token threshold is crossed
    - Periodic summarization every N messages
    - OpenAI Prompt Caching support for static system instructions
    """
    
    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        max_tokens: int = 8000,  # Token limit threshold
        summarize_every_n: int = 5,  # Summarize every N messages
        cache_system_prompt: bool = True,  # Enable prompt caching
        system_prompt: str = "You are a helpful AI assistant."
    ):
        self.model_name = model_name
        self.max_tokens = max_tokens
        self.summarize_every_n = summarize_every_n
        self.cache_system_prompt = cache_system_prompt
        
        # Initialize model
        # Note: OpenAI prompt caching is handled automatically when system prompts
        # are consistent across requests. We ensure this by always using the same
        # system message structure.
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=0.7
        )
        
        # In-memory storage
        self.chat_history: deque = deque()
        self.summaries: List[str] = []
        self.message_count = 0
        
        # System prompt (will be cached)
        self.system_prompt = system_prompt
        self.system_message = SystemMessage(content=system_prompt)
        
        # Initialize tiktoken encoder
        try:
            self.encoding = tiktoken.encoding_for_model(model_name)
        except KeyError:
            # Fallback to cl100k_base (used by gpt-4, gpt-3.5-turbo)
            self.encoding = tiktoken.get_encoding("cl100k_base")
        
        print(f"✓ MemoryManager initialized")
        print(f"  - Model: {model_name}")
        print(f"  - Max tokens: {max_tokens}")
        print(f"  - Summarize every: {summarize_every_n} messages")
        print(f"  - Prompt caching: {cache_system_prompt}")
    
    def count_tokens(self, text: str) -> int:
        """Count tokens in a text string using tiktoken."""
        return len(self.encoding.encode(text))
    
    def count_messages_tokens(self, messages: List[BaseMessage]) -> int:
        """Count total tokens in a list of messages."""
        total = 0
        for msg in messages:
            if isinstance(msg, SystemMessage):
                total += self.count_tokens(msg.content) + 4  # System message overhead
            elif isinstance(msg, HumanMessage):
                total += self.count_tokens(msg.content) + 4  # Human message overhead
            elif isinstance(msg, AIMessage):
                total += self.count_tokens(msg.content) + 4  # AI message overhead
        return total
    
    def summarize_conversation(self, messages: List[BaseMessage]) -> str:
        """
        Summarize a conversation using the LLM.
        Only summarizes user-AI pairs, not system messages.
        """
        # Extract conversation pairs (human + AI)
        conversation_text = []
        for i, msg in enumerate(messages):
            if isinstance(msg, HumanMessage):
                conversation_text.append(f"User: {msg.content}")
            elif isinstance(msg, AIMessage):
                conversation_text.append(f"Assistant: {msg.content}")
        
        conversation_str = "\n".join(conversation_text)
        
        summary_prompt = f"""Please provide a concise summary of the following conversation. 
Focus on key topics, decisions, and important information that should be remembered for future context.

Conversation:
{conversation_str}

Summary:"""
        
        summary_msg = HumanMessage(content=summary_prompt)
        response = self.llm.invoke([self.system_message, summary_msg])
        
        return response.content
    
    def should_summarize(self) -> bool:
        """Check if we should summarize based on token count or message count."""
        if len(self.chat_history) == 0:
            return False
        
        # Check token threshold
        current_tokens = self.count_messages_tokens(list(self.chat_history))
        if current_tokens > self.max_tokens:
            return True
        
        # Check message count threshold
        if self.message_count > 0 and self.message_count % self.summarize_every_n == 0:
            return True
        
        return False
    
    def add_message(self, message: BaseMessage) -> None:
        """Add a message to chat history."""
        self.chat_history.append(message)
        if isinstance(message, (HumanMessage, AIMessage)):
            self.message_count += 1
    
    def get_messages(self) -> List[BaseMessage]:
        """
        Get all messages including system prompt.
        Returns messages optimized for prompt caching.
        """
        messages = []
        
        # Add system message first (will be cached)
        if self.cache_system_prompt:
            messages.append(self.system_message)
        
        # Add summaries if any
        if self.summaries:
            summary_text = "\n\n".join([f"Previous conversation summary {i+1}: {s}" 
                                      for i, s in enumerate(self.summaries)])
            messages.append(SystemMessage(
                content=f"Previous conversation summaries:\n{summary_text}"
            ))
        
        # Add current chat history
        messages.extend(list(self.chat_history))
        
        return messages
    
    def manage_memory(self) -> Optional[str]:
        """
        Manage memory by summarizing if needed.
        Returns summary if summarization occurred, None otherwise.
        """
        if not self.should_summarize():
            return None
        
        # Get messages to summarize (excluding system message)
        messages_to_summarize = list(self.chat_history)
        
        if len(messages_to_summarize) == 0:
            return None
        
        print(f"\n📝 Summarizing conversation ({len(messages_to_summarize)} messages)...")
        
        # Create summary
        summary = self.summarize_conversation(messages_to_summarize)
        self.summaries.append(summary)
        
        # Clear chat history (summaries will be used for context)
        self.chat_history.clear()
        
        print(f"✓ Summary created ({self.count_tokens(summary)} tokens)")
        print(f"  Summary: {summary[:200]}...")
        
        return summary
    
    def chat(self, user_input: str) -> str:
        """
        Main chat method that handles memory management automatically.
        """
        # Add user message
        user_message = HumanMessage(content=user_input)
        self.add_message(user_message)
        
        # Check if we need to summarize before generating response
        summary = self.manage_memory()
        
        # Get all messages (including system prompt and summaries)
        messages = self.get_messages()
        
        # Get token count for debugging
        token_count = self.count_messages_tokens(messages)
        print(f"\n💬 User: {user_input}")
        print(f"📊 Current tokens: {token_count}/{self.max_tokens}")
        
        # Generate response
        response = self.llm.invoke(messages)
        
        # Add AI response to history
        self.add_message(response)
        
        print(f"🤖 Assistant: {response.content}")
        
        return response.content
    
    def get_memory_stats(self) -> Dict:
        """Get statistics about current memory usage."""
        current_messages = self.get_messages()
        token_count = self.count_messages_tokens(current_messages)
        
        return {
            'message_count': self.message_count,
            'chat_history_length': len(self.chat_history),
            'summary_count': len(self.summaries),
            'current_tokens': token_count,
            'max_tokens': self.max_tokens,
            'token_usage_percent': (token_count / self.max_tokens) * 100,
            'next_summarize_at': self.summarize_every_n - (self.message_count % self.summarize_every_n)
        }
    
    def clear_memory(self) -> None:
        """Clear all chat history and summaries."""
        self.chat_history.clear()
        self.summaries.clear()
        self.message_count = 0
        print("✓ Memory cleared")

# Create an instance
memory_manager = MemoryManager(
    model_name="gpt-4o-mini",
    max_tokens=8000,
    summarize_every_n=5,
    cache_system_prompt=True,
    system_prompt="You are a helpful AI assistant with excellent memory management."
)

print("\n✓ MemoryManager instance created")

✓ MemoryManager initialized
  - Model: gpt-4o-mini
  - Max tokens: 8000
  - Summarize every: 5 messages
  - Prompt caching: True

✓ MemoryManager instance created


### Basic Usage Example

Demonstrate basic memory management: the system remembers previous messages in the conversation. When asked about earlier context, it can recall information from the chat history, showing that memory is working correctly.

In [10]:
# Example: Basic usage
# The memory manager automatically handles summarization

response1 = memory_manager.chat("Hello! My name is Alice and I love machine learning.")
print("\n" + "="*50)

response2 = memory_manager.chat("What did I just tell you about myself?")
print("\n" + "="*50)

# Check memory stats
stats = memory_manager.get_memory_stats()
print("\n📊 Memory Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")


💬 User: Hello! My name is Alice and I love machine learning.
📊 Current tokens: 31/8000
🤖 Assistant: Hello, Alice! It’s great to meet you. Machine learning is a fascinating field with so much to explore. What specific aspects of machine learning are you interested in?


💬 User: What did I just tell you about myself?
📊 Current tokens: 82/8000
🤖 Assistant: You mentioned that your name is Alice and that you love machine learning.


📊 Memory Statistics:
  message_count: 4
  chat_history_length: 4
  summary_count: 0
  current_tokens: 100
  max_tokens: 8000
  token_usage_percent: 1.25
  next_summarize_at: 1


### Test Automatic Summarization by Message Count

Demonstrate automatic summarization triggered by message count. After every 5 messages (configurable), the system automatically summarizes the conversation, clears the chat history, and stores the summary for future context. This prevents unbounded memory growth.

In [11]:
# Example: Testing automatic summarization
# After 5 messages, it will automatically summarize

print("Testing automatic summarization (every 5 messages):\n")

for i in range(7):
    user_msg = f"Message {i+1}: Tell me about topic {i+1}."
    response = memory_manager.chat(user_msg)
    stats = memory_manager.get_memory_stats()
    print(f"  Messages: {stats['message_count']}, Tokens: {stats['current_tokens']}/{stats['max_tokens']}")
    print("-" * 50)

Testing automatic summarization (every 5 messages):


📝 Summarizing conversation (5 messages)...
✓ Summary created (44 tokens)
  Summary: Alice introduced herself and expressed her love for machine learning. The assistant acknowledged this information and asked about Alice's specific interests in the field. Alice then requested informat...

💬 User: Message 1: Tell me about topic 1.
📊 Current tokens: 73/8000
🤖 Assistant: Got it! Alice loves machine learning and has shown interest in a specific topic. How can I assist you further with that topic or any other questions you might have?
  Messages: 6, Tokens: 110/8000
--------------------------------------------------

💬 User: Message 2: Tell me about topic 2.
📊 Current tokens: 125/8000
🤖 Assistant: I don't have details about what "topic 2" refers to. Could you please provide more context or specify the topic you'd like to know about?
  Messages: 8, Tokens: 159/8000
--------------------------------------------------

💬 User: Message 3: Tell

### Test Token Threshold Summarization

Demonstrate summarization triggered by token count. When the conversation exceeds the token limit (set to 500 for testing), the system automatically summarizes to stay within limits. This ensures conversations don't exceed model context windows.

In [12]:
# Example: Testing token threshold summarization
# Create a manager with lower token limit to trigger summarization

test_manager = MemoryManager(
    model_name="gpt-4o-mini",
    max_tokens=500,  # Low threshold to trigger summarization
    summarize_every_n=10,  # Don't summarize by message count
    cache_system_prompt=True
)

print("Testing token threshold summarization:\n")

# Add a long conversation
long_messages = [
    "Tell me about artificial intelligence and machine learning in detail.",
    "What are neural networks? Explain in detail.",
    "Describe deep learning architectures.",
    "What is natural language processing?",
]

for msg in long_messages:
    response = test_manager.chat(msg)
    stats = test_manager.get_memory_stats()
    print(f"  Tokens: {stats['current_tokens']}/{stats['max_tokens']} ({stats['token_usage_percent']:.1f}%)")
    print("-" * 50)

✓ MemoryManager initialized
  - Model: gpt-4o-mini
  - Max tokens: 500
  - Summarize every: 10 messages
  - Prompt caching: True
Testing token threshold summarization:


💬 User: Tell me about artificial intelligence and machine learning in detail.
📊 Current tokens: 26/500
🤖 Assistant: Certainly! Artificial Intelligence (AI) and Machine Learning (ML) are closely related fields that are transforming a wide array of industries and applications. Below is a detailed overview of both concepts.

### Artificial Intelligence (AI)

**Definition:**
Artificial Intelligence refers to the simulation of human intelligence processes by machines, particularly computer systems. AI systems are designed to perform tasks that normally require human intelligence, such as reasoning, learning, problem-solving, perception, and language understanding.

**Categories of AI:**
1. **Narrow AI (Weak AI):** This type of AI is designed and trained for a specific task. Examples include virtual assistants (like Siri or 

### View Current Memory State

Display the current state of memory management: statistics about message count, token usage, summaries stored, and the current chat history. This helps monitor how the memory system is managing conversation context.

In [13]:
# Example: View current memory state
print("Current Memory State:")
print("=" * 50)

stats = memory_manager.get_memory_stats()
print("\nStatistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print(f"\nSummaries stored: {len(memory_manager.summaries)}")
for i, summary in enumerate(memory_manager.summaries, 1):
    print(f"\nSummary {i}:")
    print(f"  {summary[:200]}...")

print(f"\nCurrent chat history: {len(memory_manager.chat_history)} messages")

Current Memory State:

Statistics:
  message_count: 18
  chat_history_length: 3
  summary_count: 2
  current_tokens: 254
  max_tokens: 8000
  token_usage_percent: 3.175
  next_summarize_at: 2

Summaries stored: 2

Summary 1:
  Alice introduced herself and expressed her love for machine learning. The assistant acknowledged this information and asked about Alice's specific interests in the field. Alice then requested informat...

Summary 2:
  In the conversation, the user inquired about various topics (2 through 6), but the assistant did not have specific information on any of them. The assistant consistently requested the user to provide ...

Current chat history: 3 messages


### Custom System Prompt with Prompt Caching

Create a MemoryManager with a custom system prompt. OpenAI's prompt caching automatically caches consistent system prompts across requests, reducing token costs. The system prompt is kept consistent to enable this optimization.

In [ ]:
# Advanced: Custom system prompt with prompt caching
# The system prompt will be cached by OpenAI, reducing token costs

custom_manager = MemoryManager(
    model_name="gpt-4o-mini",
    max_tokens=8000,
    summarize_every_n=5,
    cache_system_prompt=True,  # Enable prompt caching
    system_prompt="""You are an expert data scientist and AI engineer. 
You specialize in machine learning, deep learning, and AI systems.
Always provide detailed, technical explanations with examples.
Be concise but thorough."""
)

print("✓ Custom manager created with specialized system prompt")
print("  (System prompt will be cached, reducing token usage)")

### Token Counting Demonstration

Demonstrate how `tiktoken` counts tokens in text. Shows the relationship between characters and tokens, which varies based on the text content. Token counting is essential for managing conversation length and staying within model limits.

In [ ]:
# Utility: Token counting demonstration
def demonstrate_token_counting():
    """Demonstrate how tiktoken counts tokens."""
    sample_texts = [
        "Hello, world!",
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is a subset of artificial intelligence that focuses on algorithms.",
        "In this comprehensive guide, we will explore the fundamentals of neural networks, including their architecture, training methods, and applications in various domains such as computer vision, natural language processing, and reinforcement learning."
    ]
    
    encoding = tiktoken.get_encoding("cl100k_base")
    
    print("Token Counting Demonstration:")
    print("=" * 50)
    for text in sample_texts:
        tokens = len(encoding.encode(text))
        chars = len(text)
        print(f"\nText: {text[:60]}...")
        print(f"  Characters: {chars}")
        print(f"  Tokens: {tokens}")
        print(f"  Ratio: {chars/tokens:.2f} chars/token")

# Uncomment to run:
# demonstrate_token_counting()

### Complete Workflow Example

Demonstrate the complete memory management workflow: creating a manager, having a multi-turn conversation, and observing how the system automatically summarizes at configured intervals. Shows the end-to-end behavior of the memory management system.

In [ ]:
# Complete workflow example
print("Complete Memory Management Workflow:")
print("=" * 50)

# Create a fresh manager
workflow_manager = MemoryManager(
    model_name="gpt-4o-mini",
    max_tokens=2000,  # Lower limit for demonstration
    summarize_every_n=3,  # Summarize every 3 messages
    cache_system_prompt=True
)

# Simulate a conversation
conversation = [
    "I'm working on a project about RAG systems.",
    "Can you explain how vector databases work?",
    "What's the difference between FAISS and Chroma?",
    "How do I implement semantic search?",
    "What are embeddings and how are they created?",
]

for i, user_input in enumerate(conversation, 1):
    print(f"\n--- Turn {i} ---")
    response = workflow_manager.chat(user_input)
    stats = workflow_manager.get_memory_stats()
    print(f"📊 Stats: {stats['message_count']} msgs, {stats['current_tokens']} tokens, {stats['summary_count']} summaries")

print("\n" + "=" * 50)
print("Final Memory State:")
final_stats = workflow_manager.get_memory_stats()
for key, value in final_stats.items():
    print(f"  {key}: {value}")